<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/images/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>
<div style="background-color: #1A5276; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">MLU: Building Applications with Foundation Models</h1>
    <h2 style="color: white; margin-top: 15px;">Lab 4c: Multimodal Agents</h2>
</div>

<!-- Table of Contents with All Section Levels -->
<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin-bottom: 30px;">
    <h2 style="color: #2874A6; border-bottom: 1px solid #2874A6; padding-bottom: 5px;">Table of Contents</h2>
    <p><a href="#section1" style="color: #2E86C1; font-weight: bold; text-decoration: none;">1. Installing dependencies</a></p>
    <p><a href="#section2" style="color: #2E86C1; font-weight: bold; text-decoration: none;">2. Multimodal agent for image generation and description</a></p>
    <ul style="margin-top: 0; padding-left: 30px;">
        <li><a href="#section2-1" style="color: #3498DB; text-decoration: none;">2.1 Custom multimodal tools for image generation and description</a></li>
        <li><a href="#section2-2" style="color: #3498DB; text-decoration: none;">2.2 Agentic application for image generation and description</a></li>
    </ul>
    <p><a href="#section3" style="color: #2E86C1; font-weight: bold; text-decoration: none;">3. Multimodal agent for retrieval-based responses</a></p>
    <ul style="margin-top: 0; padding-left: 30px;">
        <li><a href="#section3-1" style="color: #3498DB; text-decoration: none;">3.1 Custom multimodal tools for retrieval</a></li>
        <li><a href="#section3-2" style="color: #3498DB; text-decoration: none;">3.2 Agentic application based on retrieval</a></li>
    </ul>
    <p><a href="#section4" style="color: #2E86C1; font-weight: bold; text-decoration: none;">4. Quizzes</a></p>
</div>

In this lab, we will develop custom multimodal tools and agents to accomplish certain complex tasks.    

Multimodal agents are systems that can perceive, process, and interact with information from multiple modalities, such as text, images, audio, and video. These agents leverage various tools and APIs to plan and execute actions to achieve specific goals, combining different modalities in a seamless and intelligent manner. By combining multiple modalities, these agents can develop a more comprehensive and contextual understanding of their environment, enabling them to perform complex tasks more effectively.

The examples provided in this lab illustrate how multimodal agents can leverage different tools and APIs to achieve specific goals. In the first example, the agent uses an image generator tool to create a movie poster and then employs a multimodal model like Claude3 to generate a story based on the visual elements of the poster. In the second example, the agent performs a web search to gather visual and textual information about a product, and then uses computer vision techniques to verify the authenticity of a physical product by comparing its visual properties with the gathered information.

These examples showcase the versatility and power of multimodal agents, as they can seamlessly integrate information from different modalities, such as images and text, to perform complex tasks like creative storytelling or product verification.

<!-- Compact Lab Introduction with Activity/Challenge Explanation -->
<div style="background-color: #F8F9F9; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <h4 style="color: #2E4053; margin-top: 0;">About This Lab</h4>
    <p>Throughout this lab, you will encounter two types of interactive elements:</p>
    <table style="width: 100%; border-collapse: collapse; margin: 15px 0;">
        <tr>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/images/activity.png" alt="Activity" width="125"/>
            </td>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="125"/>
            </td>
        </tr>
        <tr>
            <td style="text-align: center; padding: 10px; background-color: #EBF5FB;">
                <p>No coding is needed for an activity. You try to understand a concept, <br/>answer questions, or run a code cell.</p>
            </td>
            <td style="text-align: center; padding: 10px; background-color: #FEF9E7;">
                <p>Challenges are where you test your understanding by implementing something new or taking a short quiz.</p>
            </td>
        </tr>
    </table>
    <p>Please work through this notebook from top to bottom to avoid errors due to missing code or context.</p>
</div>

<!-- Section Header -->
<div id="section1" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">1. Installing dependencies</h2>
</div>

In [ ]:
%%capture
!pip install -q -r ../requirements.txt

Let's import the libraries and modules required for this lab. We will import the `invoke_claude_3_multimodal` and `get_base64_encoded_image` functions we defined and used in previous labs.

In [ ]:
import sys
sys.path.append('..')

import boto3
import base64
import json
import time
from tqdm import tqdm
from botocore.exceptions import ClientError
from IPython.display import Image, display, Markdown, IFrame
import ipywidgets as widgets
from langchain.tools import tool, BaseTool
import io
from PIL import Image as pil_image,  ImageDraw, ImageFont

from mlu_utils.multimodal_utils import invoke_nova_lite_multimodal, prepare_image, get_base64_encoded_image

<!-- Section Header -->
<div id="section2" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">2. Multimodal agent for image generation and description</h2>
</div>

Let's see how we can develop a multimodal agent with custom tools for an engaging movie poster and story generation application.

1. **Movie poster generation**: You provide a prompt or concept for a movie, and the application utilizes the `image-generator-tool` to create an initial movie poster based on your input. This tool leverages the Titan Image Generator model to produce a visually compelling movie poster.

2. **Poster variation**: Once the first movie poster is generated, the `image_variation_tool` is employed to create a variation of the initial poster. This tool also uses the Titan Image generator to produce a slightly different version of the movie poster, potentially representing a different genre, mood, or style.

3. **Story generation**: With the two movie posters in hand, the application then utilizes the `Image-to-story tool`, which is powered by the Claude3 multimodal model. The multimodal agent analyzes the visual elements, symbolism, and imagery present in the movie posters and generates a compelling story or plot synopsis based on its understanding of the visual cues.

4. **Output**: The final output of the application is a set of two visually distinct movie posters and a corresponding story or plot synopsis that captures the essence and narrative suggested by the imagery. This combination of visual and language generation capabilities allows you to explore creative concepts and see how they might translate into compelling movie ideas.

The application leverages the strengths of different tools and models, including the Titan Image Generator for visual generation and the Claude3 model for visual understanding and language generation. By combining these capabilities, the application offers a unique and engaging experience for you to explore movie ideas and see how visual elements can inspire and shape narratives.

<!-- Sub-subsection Header -->
<div id="section2-1" style="border-left: 2px solid #AED6F1; padding-left: 10px; margin: 20px 0 15px 30px;">
    <h4 style="color: #3498DB;">2.1 Custom multimodal tools for image generation and description</h4>
</div>

In [ ]:
@tool
def image_to_story_tool(image_path: str):
    """Use this tool to generate a story related to a given image. The input of the tool is the path of the image."""
    #image_string, image_type = get_base64_encoded_image(image_path.replace("\n", ""))
    image_binary, image_type = prepare_image(image_path.replace("\n", ""))
    prompt = "Write an interesting story related to the given image. Produce the response without a preamble. Just write the story."
    response = invoke_nova_lite_multimodal(prompt=prompt, images=image_binary, image_types=image_type) 
    return response

In [ ]:
@tool
def add_text_to_image(image_path_and_text):
    """Use this tool to add the title to the movie poster. The input is a string with the image_path and title separated by comma."""
    # Open the image
    image = pil_image.open(image_path)

    # Create a drawing object
    draw = ImageDraw.Draw(image)

    # Define the font and its properties
    font_path = "content/Agents/FranklinGothic.ttf"  # Replace this with the path to your desired font file
    font_size = 80  # Adjust the font size as needed
    font = ImageFont.truetype(font_path, font_size)

    # Calculate the text position
    text_width = draw.textlength(text, font)
    image_width, image_height = image.size
    text_x = (image_width - text_width) / 2  # Center the text horizontally
    text_y = image_height - font_size - 50  # Position the text near the bottom

    # Draw the text on the image
    draw.text((text_x, text_y), text, font=font, fill=(255, 255, 255))  # White text color

    # Save the modified image
    image.save(output_path)

In [ ]:
@tool
def image_generator_tool(prompt: str) -> str:
    """
    Generate an image using a text prompt.
    
    Args:
        prompt (str): The text prompt for image generation.
        
    Returns:
        str: The file path of the generated image.
    """
    
    # Initialize AWS client for Bedrock Runtime
    client = boto3.client(service_name="bedrock-runtime")
    
    # Set request headers
    accept = "application/json"
    content_type = "application/json"
    
    # Set model ID for image generation
    model_id = 'amazon.nova-canvas-v1:0'
    
    # Prepare request body
    body = json.dumps({
        "taskType": "TEXT_IMAGE",
        "textToImageParams": {
            "text": prompt
        },
        "imageGenerationConfig": {
            "numberOfImages": 1,
            "height": 1024,
            "width": 1024,
            "cfgScale": 8.0,
            "seed": 0
        }
    })
    
    # Invoke the model
    response = client.invoke_model(
        body=body, modelId=model_id, accept=accept, contentType=content_type
    )
    
    # Parse the response
    response_body = json.loads(response.get("body").read())
    img = response_body.get('images')[0]
    base64_bytes = img.encode('ascii')
    image_bytes = base64.b64decode(base64_bytes)
    
    # Save the generated image
    image_path = "content/Agents/generated_image.png"
    pil_image.open(io.BytesIO(image_bytes)).save(image_path)
    
    return image_path

In [ ]:
@tool
def image_variation_tool(prompt: str):
    """
    Produce a variation of the input image based on the text prompt.

    Args:
        prompt (str): The text prompt for image variation.

    Returns:
        str: The file path of the generated variation image.
    """

    # Initialize the AWS Bedrock Runtime client
    client = boto3.client(service_name="bedrock-runtime")

    # Set the request headers and parameters
    accept = "application/json"
    content_type = "application/json"
    image_path = 'content/Agents/generated_image.png'
    model_id = 'amazon.nova-canvas-v1:0'

    # Get the base64-encoded input image
    image_string, image_type = get_base64_encoded_image(image_path)

    # Create the request body for image variation
    body = json.dumps({
        "taskType": "IMAGE_VARIATION",
        "imageVariationParams": {
            "text": prompt,
            "negativeText": "bad quality, low resolution, cartoon",
            "images": image_string,
            "similarityStrength": 0.7,  # Range: 0.2 to 1.0
        },
        "imageGenerationConfig": {
            "numberOfImages": 1,
            "height": 512,
            "width": 512,
            "cfgScale": 8.0
        }
    })

    # Invoke the Bedrock Runtime model for image variation
    response = client.invoke_model(
        body=body, modelId=model_id, accept=accept, contentType=content_type
    )
    response_body = json.loads(response.get("body").read())

    # Extract the generated image from the response
    img = response_body.get('images')[0]
    base64_bytes = img.encode('ascii')
    image_bytes = base64.b64decode(base64_bytes)

    # Save the generated image to a file
    output_path = "content/Agents/variation_image.png"
    pil_image.open(io.BytesIO(image_bytes)).save(output_path)

    return output_path

<!-- Sub-subsection Header -->
<div id="section2-2" style="border-left: 2px solid #AED6F1; padding-left: 10px; margin: 20px 0 15px 30px;">
    <h4 style="color: #3498DB;">2.2 Agentic application for image generation and description</h4>
</div>

Let's define the custom agent that will select the best tool at each planning step using the ReAct logic and accomplish the task. The application utilizes LangChain's Agent Executor to orchestrate the custom agentic workflow that leverages three tools: an image generator, an image variation tool, and an image-to-story tool.

The agent workflow proceeds as follows:

1. Ingest the user's movie concept or prompt.
2. Invoke the image generator tool to create an initial movie poster.
3. Call the image variation tool to generate a second poster variation.
4. Utilize the image-to-story tool (powered by a multimodal model) to analyze the visual elements of both posters and generate a corresponding plot synopsis.
5. Collate and present the two movie posters and the generated plot synopsis as the final output.

The agent executor acts as the central coordinator, managing the execution flow and data transfer between the custom tools. This agentic approach, facilitated by LangChain, enables the seamless integration and orchestration of the custom tools, resulting in a streamlined process for generating movie posters, variations, and narratives based on the user's input.

In [ ]:
# define custom agent
def create_custom_agent(tools):
    """
    Creates a custom agent with the given tools and a specific prompt template.

    Args:
        tools (list): A list of tools to be used by the agent.

    Returns:
        AgentExecutor: An instance of the AgentExecutor class with the custom agent.
    """
    from langchain_aws import ChatBedrockConverse
    from langchain.agents import AgentExecutor, create_react_agent
    from langchain_core.prompts.chat import ChatPromptTemplate
    
    # Initialize the large language model (LLM) with the specified model ID and temperature
    llm = ChatBedrockConverse(
        model="mistral.mistral-small-2402-v1:0",
        temperature=0
    )

    # Define the prompt template for the agent
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """Answer the following questions as best you can. You have access to the following tools:

                {tools}

                Use the following format:

                Question: the input question you must answer
                Thought: you should always think about what to do
                Action: the action to take, should be one of [{tool_names}]
                Action Input: the input to the action
                Observation: the result of the action
                ... (this Thought/Action/Action Input/Observation can repeat N times)
                Thought: I now know the final answer
                Final Answer: the final answer to the original input question""",
            ),
            ("user", "Begin!\n\nQuestion: {input}\nThought:{agent_scratchpad}")
        ]
    )

    #########################

    # Create the custom agent using the LLM, tools, and prompt
    agent = create_react_agent(llm, tools, prompt)

    # Create an instance of the AgentExecutor with the custom agent
    return AgentExecutor(agent=agent, tools=tools, verbose=True, return_intermediate_steps=True)

In [ ]:
# Create a list of all relevant tools
tools = [image_to_story_tool, image_generator_tool, image_variation_tool]

# Define the custom agent and provide the agent access to the above tools
movie_agent = create_custom_agent(tools)

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Try it yourself!</h4>
        <p>Try different prompts and observe the posters and the plots of the movies using the custom agent.</p>
    </div>
</div>

In [ ]:
# Test out the agent with a simple example
prompt = """Draw a poster for a sci-fi PG-13 movie called 'Paradox', \
make a second poster for the movie with the same title and \
then write the story of the movie based on the first poster."""

response_movie = movie_agent.invoke({"input": prompt})

#### Here's the first movie poster:

In [ ]:
Image("content/Agents/generated_image.png", width=300)

#### Here's the second movie poster:

In [ ]:
Image("content/Agents/variation_image.png", width=300)

#### And here's the plot of the movie:

In [ ]:
Markdown("<i>"+ response_movie['intermediate_steps'][-1][1] +"</i>")

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Use all the tools</h4>
        <p>In the example above, we did not need to use the <code>add_text_to_image</code> tool. Try to craft a prompt so that this tool is also used.</p>
    </div>
</div>

In [ ]:
### Enter your code below



###

<!-- Section Header -->
<div id="section3" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">3. Multimodal agent for retrieval-based responses</h2>
</div>

Here is a description of the multimodal agent for retrieval-based workflows:

This multimodal AI agent is designed to assist in verifying the authenticity of physical products by leveraging web search capabilities and advanced multimodal techniques. The agent has access to three custom tools, discussed in the next section.

<!-- Sub-subsection Header -->
<div id="section3-1" style="border-left: 2px solid #AED6F1; padding-left: 10px; margin: 20px 0 15px 30px;">
    <h4 style="color: #3498DB;">3.1 Custom multimodal tools for retrieval</h4>
</div>

Let's define three custom tools to allow the agent to retrieve results from websites to authenticate the product in the image as well as suggest websites where the authentic image may be purchased.

1. **Image Comparison Tool**: This tool utilizes Claude3's state-of-the-art multimodal capabilities to compare two product images in detail. It can analyze and compare various visual properties, such as shape, color, design elements, and intricate details, to determine the similarity or dissimilarity between the images.

2. **Product Web Search**: This tool allows the agent to perform comprehensive web searches using DuckDuckGo to gather information and visual representations of a specific product. It can retrieve product descriptions, specifications, and images from various online sources, building a comprehensive knowledge base about the product.

3. **Image Web Search**: This tool enables the agent to search the web for images of a product based on a text prompt. It can find and retrieve relevant product images from various online sources, further enhancing the agent's visual knowledge base.

In [ ]:
@tool
def image_comparison(image_paths:str):
    """Use this tool to compare and contrast two images. The input of the tool is a string consisting of both image paths seperated by comma. Image paths can be local paths or urls."""
    image_paths_arr = [f.strip().replace("\n", "") for f in image_paths.split(',')]
    try:
        image_binary, image_type = prepare_image(image_paths_arr)
    except ValueError as e:
        return f"Error loading image: {e}. Try searching for a different image URL."
    prompt = "Compare and contrast the two images. Share insights on if they are completely identical, similar or distinct. Produce the response without a preamble. Just write the analysis."
    response = invoke_nova_lite_multimodal(prompt=prompt, images=image_binary, image_types=image_type)

    return response

In [ ]:
from ddgs import DDGS

@tool
def product_web_search(prompt:str):
    """Search online for a website about a product. The input is the prompt with the product ID and the brand name. The prompt needs to be under 45 characters."""
    search_tool = DDGS()
    time.sleep(1)  # Add delay between requests
    response = search_tool.text(query=prompt, max_results=5, region='us-en', safesearch='on')
    
    return response


@tool
def image_web_search(prompt:str):
    """Search the web for images of a product based on the prompt. The input is the prompt or query used to search for images of a product."""
    search_tool = DDGS()
    time.sleep(10)  # Add delay between requests
    response = search_tool.images(query=prompt, region='us-en', max_results=1)[0]['image'].partition("?")[0]
    time.sleep(10)  # Add delay between requests
    return response

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Create custom tools</h4>
        <p>The agent's ability to pick the appropriate tool is crucial in achieving the desired response. Let's explore this ability by providing the agent with many more tools. Create a few more useful tools in the cell below and test the agent's response when it has to select from many more tools.</p>
    </div>
</div>

In [ ]:
### Enter your code below



###

<!-- Sub-subsection Header -->
<div id="section3-2" style="border-left: 2px solid #AED6F1; padding-left: 10px; margin: 20px 0 15px 30px;">
    <h4 style="color: #3498DB;">3.2 Agentic application based on retrieval</h4>
</div>

Let's define the custom agent, similar to the previous example, that will select the best tool at each planning step using the ReAct logic. 

The multimodal agent's workflow is as follows:

1. When presented with an image of a product, the agent utilizes the image web search tool to find additional images of the product, expanding its visual knowledge base.

2. Using the image comparison tool, the agent compares the visual information gathered from the web with the physical product in question. 

3. Based on the comparison results, the agent can provide an assessment of whether the physical product is likely to be authentic or an imitation.

4. Finally it will search online for a website where the user may purchase the authenticated product.

This multimodal agent leverages the power of web search, computer vision, and multimodal analysis to provide a comprehensive solution for product authentication. By combining textual and visual information from various online sources with advanced image comparison techniques, the agent can assist in verifying the authenticity of physical products with a high degree of accuracy and reliability.

In [ ]:
retrieval_tools = [product_web_search, image_web_search, image_comparison]
retrieval_agent = create_custom_agent(retrieval_tools)

In [ ]:
Image("content/Agents/nike.jpg", width=300)

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Try it yourself!</h4>
        <p>Try different images of products and observe how the agent authenticates the product using the tools.</p>
    </div>
</div>

If you get a `RateLimitException`, wait a few minutes before trying again. DuckDuckGO is a free web search API and limits the number of API requests.

Also, as it involves searching the web, sometimes it can run into error accessing some resource. If it happens, try executing the cell below agin.

In [ ]:
prompt = """I have an image of a shoe at content/Agents/nike.jpg. \ 
Can you check if they are Nike Air Jordan 1 Low SE FN5214-131? \
If they are, find the link to the website where i can find the product."""

response_shoes = retrieval_agent.invoke({"input":prompt})

#### Here's the response about the authenticity of the product:

### ⚠️ IMPORTANT SECURITY NOTICE ⚠️
The URLs displayed in this educational notebook are REAL URLs. While they are shown for educational purposes, we strongly advise:

- DO NOT click on or visit these URLs
- DO NOT use them for further exploration
- DO NOT assume they are safe or vetted

This notebook is for demonstration purposes only. Visiting unknown URLs can expose you to security risks, malware, or inappropriate content. Always practice safe browsing habits and only visit trusted, verified websites.

If you need to explore web resources, please use official documentation and trusted sources.

In [ ]:
Markdown("<i>"+response_shoes['output'] + "</i>")

#### Here's the response about the webpage to purchase the original product.

In [ ]:
url = response_shoes['intermediate_steps'][-1][1]

print(url)

<!-- Section Header -->
<div id="section4" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">4. Quiz questions</h2>
</div>

Well done on completing the lab! Now, it's time for a brief knowledge assessment.

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Try it Yourself!</h4>
        <p>Answer the following questions to test your understanding of using multimodal models for generating personalized and inclusive content.</p>
    </div>
</div>

In [ ]:
from mlu_utils.quiz_questions import lab5c_question1, lab5c_question2

lab5c_question1.display()
lab5c_question2.display()

<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin: 30px 0;">
    <h3 style="color: #2874A6; border-bottom: 1px solid #85C1E9; padding-bottom: 5px;">Conclusion</h3>
    <p>In this lab, you have:</p>
    <ul>
        <li>Developed custom multimodal tools for image generation, variation, and story creation</li>
        <li>Created a multimodal agent for creative movie poster and story generation</li>
        <li>Built custom tools for web search and image comparison</li>
        <li>Implemented a product authentication agent using retrieval-based techniques</li>
        <li>Learned how to orchestrate multiple tools using LangChain's Agent Executor</li>
    </ul>
    <h4 style="color: #2874A6; margin-top: 15px;">Additional Resources</h4>
    <ul>
        <li><a href="https://python.langchain.com/docs/modules/agents/">LangChain Agents Documentation</a></li>
        <li><a href="https://docs.aws.amazon.com/bedrock/latest/userguide/titan-image-models.html">Amazon Titan Image Generator Documentation</a></li>
        <li><a href="https://python.langchain.com/docs/modules/agents/tools/custom_tools">Creating Custom Tools in LangChain</a></li>
    </ul>
</div>

<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/images/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>

# Thank you!